# Lab 03: The ReAct Pattern for Step-by-Step Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab03_react_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

### What this lab is about

In Lab 02 the AI could use **tools**: small helpers that each look up one fact in the evidence.
In this lab the AI must also **say why** before every lookup. It works in a loop, the way a
careful examiner works through a case:

1. **Thought:** "What do I need to know next, and why?"
2. **Action:** use one tool to look it up in the evidence.
3. **Observation:** read what the tool found.

It repeats this loop until it has enough evidence to answer. The loop is called **ReAct**, short
for *Reason + Act*.

**The question you will answer:** while a phone was left unattended, did someone try to send an
image through Signal, and what can the network records tell us about whether it was delivered?

### How this notebook flows

- **Part 0:** set up. Install one library, connect to a free AI model, write the case files, and
  load the ReAct toolbox.
- **Part A:** you run the loop by hand, one turn per cell, so you can see every Thought, Action,
  and Observation.
- **Part B:** a ready-made agent runs the same loop on its own, and you check its work.
- **Your answer:** you write your own evidence-based answer and compare the two runs.

Run the cells from top to bottom, in order. Each step uses what the step before it made.

The AI is a helper, not the decision maker. It can be wrong. You are the examiner: you make the
final call, and every claim must point back to a piece of evidence.

## Part 0: One-Time Setup

These four steps get the notebook ready. They install one library, connect you to a **free
AI model** hosted online by a company called Groq, write the case files, and load the ReAct
toolbox. Nothing is installed on your own computer.

### Step 0.1: Install the helper library

**Purpose:** the notebook needs one small add-on, the `openai` library, to send questions to the
AI and bring the answers back. Groq accepts the same format, so this library works with it.

**Input:** nothing from you. The library is downloaded into this online session only.

**How it works:** `%pip install` is a notebook command that installs a Python library. The `-q`
part keeps the install log short.

**Output:** a short install log. Warnings are normal. Step 0.2 uses the library to connect to the AI.

In [ ]:
# Install the one library this notebook needs. It goes into this online session only.
%pip install -q openai

### Step 0.2: Connect to a free AI model (Groq)

**Purpose:** connect the notebook to the AI model that will do the reasoning in this lab.

**Input:** your free Groq **API key** (a private password that lets the notebook use the AI).
To get one:

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell below and paste the key when asked. It stays hidden and is **not saved** into the
   notebook.

**How it works:** the cell asks for your key without showing it, opens a connection to Groq, and
picks the first AI model from a short list that your key can use. It also adds a small safety
helper that keeps each reply short enough for the free plan and waits and tries again when the
service is busy. Finally, it sends a one-word test question.

**Output:** a line that starts with `Model OK:` and shows the model's name and its one-word reply.
This step creates `client` (the connection to the AI) and `MODEL` (the name of the AI model in
use). Every AI step from Step A.2 onward uses these two. If you see a list of model names and an
error instead, follow the message: put one of the listed names first in `PREFERRED` and run the
cell again.

In [ ]:
# Connect this notebook to a free AI model hosted by Groq, and check that it answers.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: carries your messages to the AI and back (Groq uses this format)

# Ask for your key only if this session does not already have it.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service. Every question for the AI is sent here.
BASE_URL = 'https://api.groq.com/openai/v1'
# A short list of free AI models that work well for this lab, best first. Groq retires models
# from time to time, so the cell uses the first one your key can reach and prints its name.
# Edit the list to try another model.
# Qwen comes first here because this lab's agent must write its tool requests as plain text
# inside <tool_call> tags. Qwen does that reliably; gpt-oss tends to plan the request silently
# and send back an empty reply.
PREFERRED = ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b']

# Build the client: the connection that sends questions to the AI and brings back replies.
# max_retries=8: the free plan limits how much text you can send each minute. If you go over,
# Groq says 'try again in a few seconds', and the client waits and tries again instead of
# stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every AI request in this notebook goes through the small helper below. It does two things:
# 1. It caps the length of each reply, so you stay inside the free plan's per-minute limit.
#    Groq measures text in tokens (small pieces of words) and counts your question PLUS the
#    longest allowed reply. Replies are capped at 1,500 tokens, or 900 for Qwen, whose free plan
#    allows only 1,000 reply tokens a minute. (Any single request can still set its own cap.)
# 2. The gpt-oss model sometimes tries a built-in shortcut instead of writing plain text, and Groq
#    rejects that ('model called a tool'). The helper then asks again, up to 3 tries in total,
#    with a reminder to reply in plain text.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
# From now on, every AI request made through the client uses the helper above.
client.chat.completions.create = _create_within_limits

# Ask Groq which models your key can use, then pick the first match from PREFERRED.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send one tiny test question, so you know the connection works before the case starts.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the case evidence files

**Purpose:** put the case evidence on disk so the tools can read it, the same way an examiner
works from files exported off a phone.

**Input:** the five evidence files for case `RA-2026-009`, stored inside the cell below as text.
They are synthetic (made up for teaching) and match the copies in the course repository:

- `artifact_manifest.json`: a summary of the case: the device, how it was acquired, the hash of
  the original image, and the reported unattended time.
- `incident_window.csv`: when the unattended time started and ended, from a staff observation log.
- `messaging_events.csv`: the Signal image-attachment attempt.
- `network_events.csv`: when the phone's mobile data dropped and when it came back.
- `chain_of_custody.csv`: who handled the evidence, and when.

**How it works:** the cell makes a folder called `data/` and writes each file into it. A file that
is already there is left alone, never overwritten.

**Output:** one line per file, for example `wrote data/incident_window.csv`. (If a file is already
there from an earlier run, it is left unchanged and its line says `exists`.) The tools in Step A.1 read three of
these files: the incident window, the messaging events, and the network events.

In [ ]:
# Write the five case evidence files into a folder called data/, so the tools can read them later.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

# ARTIFACTS is a dictionary (a list of name -> value pairs): each file name -> the text inside it.
ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "RA-2026-009",\n  "lab": "Lab 03 - ReAct Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Samsung Galaxy A54",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-02-21T10:43:18Z",\n    "original_image_sha256": "a7a4db1c2e1d75889f847d0b46a60bf8b84ae4d1533f6f5ce7aa98fb29dcb6ef"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-20T14:10:00Z",\n    "end": "2026-02-20T14:25:00Z"\n  },\n  "interval_source": "staff observation log",\n  "note": "Synthetic classroom dataset for react-pattern demo built around a reported unattended interval"\n}\n', 'incident_window.csv': 'timestamp_utc,event,source,notes\n2026-02-20T14:10:00Z,unattended_interval_start,staff_observation,phone left on supply table before coordinator stepped away\n2026-02-20T14:25:00Z,unattended_interval_end,staff_observation,phone recovered from supply table when coordinator returned\n', 'messaging_events.csv': 'timestamp_utc,app,event,contact,details\n2026-02-20T14:16:11Z,Signal,attachment_attempt,+1-555-0179,outgoing image attachment attempt recorded while mobile data unavailable\n', 'network_events.csv': 'timestamp_utc,event,status,details\n2026-02-20T14:15:58Z,connectivity_change,offline,mobile data dropped during reported unattended interval\n2026-02-20T14:28:02Z,connectivity_change,restored,mobile data restored after reported unattended interval\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-21T10:43:18Z,Analyst Rivera,acquisition,full file system collected from outreach phone\n2026-02-21T11:05:44Z,Analyst Rivera,hash_verification,original image hash verified before review\n'}

# Make the data/ folder. exist_ok=True means: no error if the folder is already there.
data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
# A loop: repeat the indented lines once for each file. Only missing files are written.
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if fp.exists():
        print('exists ' + str(fp))   # already there from an earlier run: left unchanged
    else:
        fp.write_text(text)
        print('wrote ' + str(fp))    # a new file, just written

### Step 0.4: Load the ReAct toolbox

**Purpose:** load the ready-made code that powers this lab's agent. Think of it as the toolbox the
rest of the notebook reaches into.

**Input:** nothing from earlier steps. All of the code is inside the cell below.

**How it works:** the cell defines four main things. You do not need to read every line.

- `@tool`: a label that turns a normal Python function into a tool the AI is allowed to ask for.
- `extract_tag_content`: the AI marks each part of its reply with **tags**, such as
  `<thought>...</thought>`, `<tool_call>...</tool_call>`, and `<response>...</response>`. This
  helper pulls out the text between a pair of tags, so the code can find the Thought, the tool
  request, or the final answer.
- `REACT_SYSTEM_PROMPT`: the standing instructions (the **prompt**: the instructions we give the AI)
  that teach the AI how to run the Thought, Action, Observation loop.
- `ReactAgent`: a packaged agent that runs the whole loop on its own.

This copy is adapted from The Neural Maze's agentic-patterns-course (MIT License).

**Output:** nothing is printed. Step A.1 uses `@tool`, Step A.2 uses the instructions and the tag
helper, and Step B.1 uses `ReactAgent`.

In [ ]:
# The ReAct toolbox: the code that powers this lab's agent. Run it once and move on.
# You do NOT need to read every line. Skim the names: tool, extract_tag_content,
# REACT_SYSTEM_PROMPT (the AI's standing instructions), and ReactAgent (used in Part B).
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

import json
import re
from dataclasses import dataclass
from typing import Callable


# ---------------------------------------------------------------- 1. reading the AI's reply
# The AI marks each part of its reply with tags, for example <thought>...</thought>.
# extract_tag_content finds the text between a pair of tags, so the code can use that part.

# What extract_tag_content hands back: the pieces of text it found, and whether it found any.
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Extract everything enclosed by <tag>...</tag> (used for <thought>, <tool_call>, <response>)."""
    # Some models repeat the closing </tool_call> tag at the end; tidy that up first.
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    # Find every <tag>...</tag> pair and keep the text inside, without extra spaces.
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- 2. the memory (chat history)
# Each message in the memory is a small dictionary: who is speaking ('system', 'user', or
# 'assistant', meaning the AI) and what they said ('content').
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


# A list of messages that can forget its oldest message once it reaches a size limit.
# total_length=-1 means "no limit", which is what this lab uses, so nothing is ever forgotten.
class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


# Send the memory to the AI and return just the text of its reply.
def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- 3. turning a function into a tool
# get_fn_signature writes a short description card for a function: its name, its description
# (the text in triple quotes under it), and the inputs it takes. The AI reads these cards to
# decide which tool to ask for.
def get_fn_signature(fn: Callable) -> dict:
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


# Check the inputs the AI asked for: drop any the tool does not take, and convert each one to the
# type the tool expects (for example, the text "5" into the number 5).
def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


# A Tool keeps the function together with its name and description card. run() calls the function.
class Tool:
    def __init__(self, name, fn, fn_signature):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """Decorator: turn a plain function into a Tool (its name, docstring, and typed parameters)."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- 4. the AI's standing instructions
# These instructions are sent to the AI at the start of every ReAct run. They explain the
# Thought -> Action -> Observation loop and the tags to use. The %s near the middle is a blank
# that gets filled in with the tools' description cards.
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of three steps:

1. **Thought** - Reason about the current situation and decide what to do next. Wrap in <thought></thought> tags.
2. **Action** - Call one tool by emitting a <tool_call> JSON object (see format below).
3. **Observation** - The tool result will be returned wrapped in <observation></observation> tags. Use it to inform your next thought.

Repeat this loop until you have enough information to give a final answer, then wrap it in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

- Do NOT make assumptions about parameter values.
- Pay close attention to each parameter's `type` and use the correct Python types.
- Use exactly one tool call per turn, and increment `id` for each call.

## Available Tools
<tools>
%s
</tools>

## Example Session
<question>What's the current temperature in Madrid?</question>
<thought>I need to retrieve the current weather in Madrid.</thought>
<tool_call>{"name": "get_current_weather", "arguments": {"location": "Madrid", "unit": "celsius"}, "id": 0}</tool_call>
<observation>{0: {"temperature": 25, "unit": "celsius"}}</observation>
<thought>I now have the temperature for Madrid. I can respond to the user.</thought>
<response>The current temperature in Madrid is 25 degrees Celsius.</response>

## Additional Constraints
- Do not skip directly to a final answer while important evidence is still missing.
- Keep the final answer evidence-bounded: state only what the observations support.
- Never leave the loop without eventually producing a <response>.
"""


# ---------------------------------------------------------------- 5. the packaged ReAct agent
class ReactAgent:
    """Reason -> Action -> Observation loop. The model thinks, calls one tool, reads the result,
    and repeats until it emits a <response>."""

    def __init__(self, tools, model="openai/gpt-oss-120b", client=None):
        if client is None:
            raise ValueError("ReactAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    # Run each tool the AI asked for and collect the results. A garbled request is skipped; an
    # unknown tool name or a bad input becomes a note back to the AI instead of a crash.
    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        for tool_call_str in tool_calls_content:
            try:
                tool_call = json.loads(tool_call_str)
            except json.JSONDecodeError:
                print(f"\nSkipping malformed tool call: {tool_call_str}")
                continue
            tool_name = tool_call.get("name") if isinstance(tool_call, dict) else None
            t = self.tools_dict.get(tool_name)
            if t is None:
                print(f"\nSkipping unknown tool request: {tool_name}")
                observations[len(observations)] = f"Unknown tool {tool_name!r}. Use one of: {', '.join(self.tools_dict)}."
                continue
            print(f"\nUsing Tool: {tool_name}")
            tool_call.setdefault("arguments", {})
            try:
                validated = validate_arguments(tool_call, json.loads(t.fn_signature))
                print(f"Tool call: {validated}")
                result = t.run(**validated["arguments"])
            except Exception as e:   # a wrong or missing argument becomes an observation, not a crash
                validated, result = tool_call, f"Tool error: {e}"
            print(f"Tool result: {result}")
            observations[validated.get("id", len(observations))] = result
        return observations

    def run(self, user_msg: str, max_rounds: int = 10) -> str:
        # Start a fresh memory: the standing instructions (with the tool cards) and the question.
        system_prompt = REACT_SYSTEM_PROMPT % self.add_tool_signatures()
        chat_history = ChatHistory([
            build_prompt_structure(system_prompt, "system"),
            build_prompt_structure(user_msg, "user", tag="question"),
        ])
        tools_used, pushbacks = 0, 0
        # Repeat the loop up to max_rounds times.
        for _ in range(max_rounds):
            # Ask the AI for its next move, then pull out any Thought, tool request, or answer.
            completion = completions_create(self.client, chat_history, self.model)

            thought = extract_tag_content(str(completion), "thought")
            tool_calls = extract_tag_content(str(completion), "tool_call")
            response = extract_tag_content(str(completion), "response")
            if not str(completion).strip():       # some models plan the call silently and send nothing
                update_chat_history(chat_history, "Your reply was empty. Write your <thought> and "
                                    "<tool_call> tags as plain text in the reply.", "user")
                continue
            # A <response> only counts when no tool call came with it. A model that writes a tool
            # call, imagines its result, and answers in one go must first run the tool and see the
            # REAL result.
            if response.found and not tool_calls.found:
                if self.tools_dict and tools_used == 0 and pushbacks < 2:
                    # Evidence check: an answer given before any tool has run cannot be based on
                    # the evidence, so push back and ask for a tool call first.
                    pushbacks += 1
                    update_chat_history(chat_history, completion, "assistant")
                    update_chat_history(chat_history, "You have not called any tool yet, so that answer is "
                                        "not based on evidence. Call a tool first with a <tool_call> block.", "user")
                    continue
                return response.content[0]
            # Keep only the AI's Thought and tool request. Cut off any observation or answer it made
            # up on its own, because the real result must come from running the tool.
            if tool_calls.found:
                completion = str(completion).split("<observation>")[0].split("<response>")[0]
            update_chat_history(chat_history, completion, "assistant")

            if thought.found and thought.content:
                print(f"\nThought: {thought.content[0]}")

            # Run the requested tool(s) and add the real results to the memory as an <observation>.
            if tool_calls.found:
                observations = self.process_tool_calls(tool_calls.content)
                tools_used += len(observations)
                print(f"\nObservations: {observations}")
                update_chat_history(chat_history, f"<observation>{observations}</observation>", "user")

        # Out of rounds without a final answer: ask the AI one last time and return what it says.
        return completions_create(self.client, chat_history, self.model)

## The Case in One Minute

**Case `RA-2026-009`:** an outreach worker's Samsung Galaxy A54 was left unattended on a supply
table for a short time. The question is narrow: during that time, did the phone try to send an
image through Signal, and can we say anything about whether it was delivered?

| UTC time | What the records show |
|---|---|
| 14:10:00 | unattended interval starts (staff observation) |
| 14:15:58 | mobile data drops offline |
| 14:16:11 | Signal image-attachment attempt recorded |
| 14:25:00 | unattended interval ends |
| 14:28:02 | mobile data restored (after the interval) |

(All times are UTC, a single world time standard, so records from different sources line up.)

Two words matter in this lab:

- An **attempt** means the phone *tried* to send the image. The records show this.
- A **delivery** means the image actually *reached* the other person. The records do not show this.

The attempt falls **inside** the unattended interval, but the phone was offline at that moment, and
its connection only came back **after** the interval ended. Keeping attempt and delivery apart is
the point of the lab. Saying "the image was delivered" would be an **overclaim**: a statement the
evidence does not support.

## How ReAct Works, and Where the Memory Lives

Each turn of the loop has three parts. The AI marks each part with a tag so the code can find it:

- **Thought** (`<thought>`): the AI says what it needs to know next, and why.
- **Action** (`<tool_call>`): the AI asks for exactly one tool.
- **Observation** (`<observation>`): the notebook runs that tool on the evidence and hands the
  result back to the AI.

The loop repeats until the AI writes a final answer inside **`<response>`** tags.

How does each turn build on the last? The AI does not remember anything by itself between requests.
Instead, the notebook keeps a simple **short-term memory**: a running list called `chat_history`.
It holds the instructions, the question, every Thought, every tool result, and every AI reply. Each
turn, the whole list is sent to the AI again. Because nothing is remembered outside that list, you
can read the agent's full reasoning trail from top to bottom, like a case note.

## Part A: Walk the ReAct Loop One Turn at a Time

In this part you drive the loop yourself, one turn per cell, so every Thought, Action, and
Observation is visible before you hand the case to the packaged agent in Part B.

- **Step A.1:** build the three evidence tools.
- **Step A.2:** write the instructions and the question, start the memory, and make a one-turn helper.
- **Steps A.3 to A.5:** run three turns, one tool each.
- **Step A.6:** ask the AI for its final, evidence-based answer.

### Step A.1: Build the three evidence tools

**Purpose:** give the AI three narrow tools. Each one answers one small question from the evidence.
Because each tool shows only one slice of the case, the AI has to collect the evidence step by step
instead of guessing.

**Input:** the case files in the `data/` folder from Step 0.3, and the `@tool` label from Step 0.4.

**How it works:** the cell defines three tools:

- `get_incident_window` reads `incident_window.csv` and returns when the unattended interval
  started and ended, and where each time came from.
- `get_message_attempt` reads `messaging_events.csv` and returns the first attachment attempt for
  the app you name (here, Signal).
- `get_network_restore_time` reads `network_events.csv` and returns the time mobile data came back.

The sentence in triple quotes under each tool is its **description**. The AI reads these
descriptions to decide which tool to ask for, so they are written for the AI.

**Output:** the line `Defined tools:` followed by the three tool names. The cell also creates
`available_tools`, a menu of the tools by name. Step A.2 uses this menu to run whichever tool the
AI asks for.

In [ ]:
# Build the three evidence tools. Each tool reads ONE evidence file and returns one small fact.
import csv   # csv: reads spreadsheet-style files (rows of comma-separated values)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path

data_dir = Path('data')  # the folder of case files written in Step 0.3

# Helper: open a CSV file (a spreadsheet saved as plain text) and return all of its rows.
# Each row comes back as a dictionary of column name -> value, for example 'event' -> '...'.
def read_csv(filename):
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

# @tool turns the normal function below it into a tool the AI can ask for.
@tool
def get_incident_window() -> str:
    """Return the start and end timestamps for the reported unattended interval."""
    rows = read_csv('incident_window.csv')
    # Find the row that marks the start and the row that marks the end of the unattended time.
    start = next(r for r in rows if r['event'] == 'unattended_interval_start')
    end = next(r for r in rows if r['event'] == 'unattended_interval_end')
    # Hand back both times, plus the source and the note for each, as one block of labelled text.
    return json.dumps({'start': start['timestamp_utc'], 'end': end['timestamp_utc'],
                       'start_source': start['source'], 'end_source': end['source'],
                       'start_note': start['notes'], 'end_note': end['notes']})

@tool
def get_message_attempt(app: str) -> str:
    """Return the first recorded messaging attachment attempt for the requested app."""
    rows = read_csv('messaging_events.csv')
    # Find the first row for the app the AI named. Upper or lower case does not matter.
    match = next((r for r in rows if r['app'].lower() == app.lower()), None)
    # If that app never appears in the records, say so plainly instead of crashing.
    if match is None:
        return json.dumps({'error': f'No messaging event found for {app}.'})
    return json.dumps(match)

@tool
def get_network_restore_time() -> str:
    """Return the timestamp when mobile data connectivity was restored."""
    rows = read_csv('network_events.csv')
    # Find the first row where the connection came back ('restored') and return just its time.
    restored = next(r for r in rows if r['status'] == 'restored')
    return restored['timestamp_utc']

# The tool menu: a dictionary of tool name -> tool. Step A.2 looks tools up here by name.
available_tools = {t.name: t for t in [get_incident_window, get_message_attempt, get_network_restore_time]}
print('Defined tools:', list(available_tools))

### Step A.2: Write the instructions, start the memory, and make a one-turn helper

**Purpose:** set up everything the guided walk-through needs: the AI's standing instructions, the
case question, the short-term memory, and a helper that runs exactly one turn of the loop.

**Input:** the tool menu `available_tools` from Step A.1; `REACT_SYSTEM_PROMPT` and
`extract_tag_content` from Step 0.4; and `client` and `MODEL` from Step 0.2.

**How it works:**

- It adds the three tool descriptions to the standing instructions, so the AI knows which tools
  exist and how to ask for them.
- It writes the case question as `USER_QUESTION`. In this guided version the question also tells
  the AI which order to use the tools in, so every turn is predictable and easy to check. It asks
  for a four-part final answer.
- It starts `chat_history`, the memory, with two messages: the instructions and the question.
- It defines `run_single_react_turn`. Each time you call it, it sends the memory to the AI, prints
  the reply, finds the tool request, runs that tool on the evidence, prints the result, and adds
  the result to the memory as an Observation.

**Output:** the line `Prompt and helper ready.` Nothing is sent to the AI yet. Steps A.3, A.4, and
A.5 each call the helper once, and `USER_QUESTION` is reused in Part B.

In [ ]:
# Prepare the guided walk-through: the AI's instructions, the question, the memory, and a helper.

# Put the three tool descriptions into the standing ReAct instructions from Step 0.4,
# so the AI knows which tools exist and how to ask for them.
tools_signature = ',\n'.join(t.fn_signature for t in available_tools.values())
formatted_system_prompt = REACT_SYSTEM_PROMPT % tools_signature

# The case question. In this guided version it also fixes the order of the three tools,
# so each turn is easy to check, and it spells out the four parts of the final answer.
USER_QUESTION = (
    'Determine whether someone tried to send an image through Signal during the reported unattended '
    'interval, and use network restoration evidence to avoid overstating confirmed delivery. '
    'Use exactly one tool call per turn in this order: get_incident_window, then get_message_attempt '
    'for Signal, then get_network_restore_time. After the third observation, give a final answer with: '
    'ReAct step log (one line per tool step); unattended-interval evidence (start and end timestamps); '
    'Signal image-sending timing evidence (attempt timestamp and whether it is inside the interval); '
    'connectivity context and evidence-bounded conclusion (network restore time, then one careful '
    'sentence that does not claim confirmed delivery).'
)

# chat_history is the agent's short-term memory: a list of every message so far.
# It starts with the standing instructions ('system') and the case question ('user').
# Each turn adds the AI's reply and the tool's result, so the next turn can build on them.
chat_history = [
    {'role': 'system', 'content': formatted_system_prompt},
    {'role': 'user', 'content': f'<question>{USER_QUESTION}</question>'},
]

# The one-turn helper: runs exactly one Thought -> Action -> Observation cycle.
def run_single_react_turn(history):
    # Send the whole memory to the AI and get back its next Thought and Action.
    model_response = client.chat.completions.create(messages=history, model=MODEL).choices[0].message.content
    print(model_response)
    # Add the AI's reply to the memory.
    history.append({'role': 'assistant', 'content': model_response})
    # Look for a <tool_call> block. If there is none, the AI may be giving its final answer.
    extracted = extract_tag_content(model_response, tag='tool_call')
    if not extracted.found:
        return model_response, None, None
    # Read the tool request (which tool, with which inputs), then run that tool on the evidence.
    tool_request = json.loads(extracted.content[0])
    observation = available_tools[tool_request['name']].run(**tool_request['arguments'])
    print('\nTool request:', tool_request)
    print('Observation:', observation)
    # Add the tool's real result to the memory as an <observation>, ready for the next turn.
    history.append({'role': 'user', 'content': f'<observation>{observation}</observation>'})
    return model_response, tool_request, observation
print('Prompt and helper ready.')

### Step A.3: Turn 1: find the unattended interval

**Purpose:** you cannot say whether something happened *during* the unattended time until you know
*when* that time was. The AI should work this out and ask for the time window first.

**Input:** `chat_history` from Step A.2, which holds the instructions and the question.

**How it works:** the cell runs one turn with the helper from Step A.2. The AI writes a Thought and
asks for a tool, the tool runs on the evidence, and the result is added to the memory.

**Output:** the AI's reply, with a `<thought>` and a `<tool_call>` asking for
`get_incident_window`. Then a `Tool request:` line and an `Observation:` line showing the start
`2026-02-20T14:10:00Z` and the end `2026-02-20T14:25:00Z`, both from the staff observation log.
The memory now holds this result, ready for Step A.4.

*The AI does not always word its reply the same way, and now and then it may not follow the
instructions. If the reply has no tool request, run Step A.2 again to reset the memory, then run
this step again.*

In [ ]:
# Turn 1: the AI thinks, asks for one tool, and the tool's result is added to chat_history.
model_response_1, tool_request_1, observation_1 = run_single_react_turn(chat_history)

### Step A.4: Turn 2: check the Signal attempt

**Purpose:** now that the time window is known, the AI checks whether there is a Signal image
attempt, and when it happened.

**Input:** `chat_history`, which now also holds the time window found in Step A.3.

**How it works:** the same helper runs a second turn. This time the AI should ask for
`get_message_attempt` and name Signal as the app.

**Output:** a tool request for `get_message_attempt` with the app set to `Signal`, and an
observation showing an `attachment_attempt` at `2026-02-20T14:16:11Z`, with the note that it was
recorded while mobile data was unavailable. 14:16:11 is **inside** the 14:10:00 to 14:25:00
interval. The memory now holds this result too, ready for Step A.5.

In [ ]:
# Turn 2: same helper again. The memory now includes the time window from Turn 1.
model_response_2, tool_request_2, observation_2 = run_single_react_turn(chat_history)

### Step A.5: Turn 3: check when the connection came back

**Purpose:** the Signal record says the attempt happened while mobile data was unavailable. So the
AI checks when the connection came back. This stops it from mistaking an attempt for a delivery.

**Input:** `chat_history`, which now holds the time window (Step A.3) and the Signal attempt
(Step A.4).

**How it works:** the helper runs a third turn. The AI should ask for `get_network_restore_time`.

**Output:** an observation of `2026-02-20T14:28:02Z`, the moment mobile data was restored. That is
**after** the unattended interval ended at 14:25:00. The memory now holds all three observations,
ready for Step A.6.

In [ ]:
# Turn 3: the memory now includes the time window and the Signal attempt.
model_response_3, tool_request_3, observation_3 = run_single_react_turn(chat_history)

### Step A.6: Ask for the final, evidence-based answer

**Purpose:** with all three observations gathered, the AI writes the four-part answer the question
asked for. It must not claim the image was delivered: the records show an attempt and a later
reconnect, not a confirmed delivery.

**Input:** `chat_history` from Steps A.3 to A.5: the instructions, the question, and three
observations.

**How it works:** the cell sends the whole memory to the AI one more time, without the one-turn
helper. The AI should see that it has enough evidence and write its answer inside `<response>` tags.

**Output:** the AI's final answer, saved as `final_output` and printed. It should contain a step
log, the interval (14:10:00 to 14:25:00), the attempt time (14:16:11, inside the interval), the
restore time (14:28:02, after the interval), and one careful sentence that does **not** say the
image was delivered. Check that last part yourself. If the AI asks for another tool instead of
answering, note that it did not follow the plan. You will use this answer when you compare the two
runs at the end.

In [ ]:
# Send the full memory (question + three observations) and ask for the final answer.
# .choices[0].message.content pulls the plain text of the reply out of the AI's response.
final_output = client.chat.completions.create(messages=chat_history, model=MODEL).choices[0].message.content
print(final_output)

## Part B: Run the Same Case with `ReactAgent`

In Part A you pressed the button for every turn. Now you hand the same tools and the same question
to the packaged `ReactAgent`, which runs the Thought, Action, Observation loop on its own. Your job
changes from running the loop to **checking** it: did it gather the same observations, in a
sensible order, and did it stay within what the evidence shows?

### Step B.1: Let the agent run the whole loop

**Purpose:** see how an agent works through the case without you stepping it turn by turn, and
judge whether its answer is as careful as yours.

**Input:** the three tools from Step A.1, the same `USER_QUESTION` from Step A.2, `client` and
`MODEL` from Step 0.2, and `ReactAgent` from Step 0.4.

**How it works:** the agent starts its own fresh memory with the instructions and the question,
then repeats the loop (up to 10 rounds) until the AI writes a final `<response>`. It has some
built-in safety checks:

- **Evidence check:** if the AI tries to answer before using any tool, the agent pushes back and
  tells it to use a tool first (up to two times), because an answer with no evidence behind it
  is a guess.
- If the AI asks for a tool that does not exist, or gives a tool the wrong input, the agent tells
  the AI about the problem instead of crashing.
- If 10 rounds pass without a final answer, the agent asks the AI one last time and returns
  whatever it says.

**Output:** for each round, lines starting with `Thought:`, `Using Tool:`, `Tool call:`,
`Tool result:`, and `Observations:`, then the agent's final answer at the bottom. This run does not
use or change your `chat_history` from Part A. You will compare its answer with yours below.

In [ ]:
# Hand the same three tools and the same question to the packaged agent.
react_agent = ReactAgent(
    tools=[get_incident_window, get_message_attempt, get_network_restore_time],
    client=client, model=MODEL)

# run() repeats Thought -> Action -> Observation on its own and returns the final answer.
print(react_agent.run(user_msg=USER_QUESTION))

## Your Evidence-Bounded Answer

**Purpose:** write your own answer, as the examiner. The AI's answers above are drafts to check,
not the final word.

**Input:** the observations you collected in Steps A.3 to A.5, and what you saw in Part B. Base
every line on those observations, not on general knowledge.

**How it works:** the cell below holds your answer in four labelled parts. Replace each `'...'` with
your own words, and replace `None` with `True` or `False`, then run the cell.

1. **ReAct step log:** one short line per tool step.
2. **Unattended-interval evidence:** the start and end timestamps.
3. **Signal image-sending timing evidence:** the attempt timestamp, and whether it is inside the
   interval (`True` or `False`).
4. **Connectivity context and conclusion:** the network restore time, then one careful sentence that
   does **not** claim confirmed delivery.

**Output:** your four parts printed, one per line. They are saved with the notebook when you submit.

In [ ]:
# Your answer. Replace each '...' with your own words, and None with True or False.
answer = {
    'react_step_log': ['1 get_incident_window ...', '2 get_message_attempt ...', '3 get_network_restore_time ...'],
    'unattended_interval': {'start': '...', 'end': '...'},
    'signal_timing': {'attempt_utc': '...', 'inside_interval': None},
    'connectivity_and_conclusion': {'restored_utc': '...', 'conclusion': '...'},
}
# Print each part of your answer on its own line, so you can read it back.
for k, v in answer.items():
    print(k, '->', v)

### Compare the two runs

Look back at your hand-driven run (Part A) and the agent's run (Part B), then answer:

1. Did `ReactAgent` gather the same three observations you did, in a sensible order?
2. Did it avoid claiming the image was **delivered**?
3. If any claim was too strong, how would you rewrite it so it matches the evidence?

## Submit

Save the notebook with Part A, Part B, and your answer filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The ReAct
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*